# Project 02

## Import helpers

Run the following $2$ cells to download and import helper functions, files and libraries.

You might not need them, but they're here just in case.

I built a motor fault detection pipeline on the CWRU bearing dataset, extracting vibration features (kurtosis, RMS, crest factor) from segmented signals and training an Isolation Forest on normal data to flag anomalies.

In [3]:
!wget -q https://github.com/PSAM-5005-2026S-A/5005-utils/raw/main/src/audio_utils.py
!wget -q https://github.com/PSAM-5005-2026S-A/5005-utils/raw/main/src/data_utils.py
!wget -q https://github.com/PSAM-5005-2026S-A/5005-utils/raw/main/src/image_utils.py

In [154]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from scipy.stats import kurtosis


## Milestone 02

Start bringing data into your project here

The dataset consists of `.mat` files representing vibration signals from different machine conditions.


In [155]:
DATA_DIR = Path("data/raw")
files = list(DATA_DIR.rglob("*.mat"))
print("Total .mat files found:", len(files))

Total .mat files found: 171


In [156]:
def load_signal(filepath):
    mat = loadmat(filepath)

  
    de_keys = [k for k in mat.keys() if 'DE_time' in k]
    if de_keys:
        return mat[de_keys[0]].flatten()

 
    for key in mat:
        if not key.startswith("__"):
            data = np.array(mat[key]).squeeze().reshape(-1)
            if len(data) > 0:
                return data

    return None

## Milestone 03

Continue pulling and organizing data here

In [157]:
def segment_signal(signal, window_size=2048, stride=512):
  
    if signal is None:
        return []

    signal = np.asarray(signal).flatten()

    if len(signal) < window_size:
        return []

    segments = []
    for start in range(0, len(signal) - window_size + 1, stride):
        segments.append(signal[start:start + window_size])

    return segments

# Feature extraction

 Extract time-domain and frequency-domain features from one window.

In [158]:
def extract_features(segment, sampling_rate=12000):
   
    segment = np.asarray(segment).flatten()
    features = {}

    rms = np.sqrt(np.mean(segment**2))
    peak = np.max(np.abs(segment))

    features["mean"]         = np.mean(segment)
    features["std"]          = np.std(segment)
    features["rms"]          = rms
    features["peak"]         = peak
    features["crest_factor"] = peak / (rms + 1e-10)
    features["kurtosis"]     = kurtosis(segment)


    fft_vals   = np.fft.rfft(segment)
    magnitude  = np.abs(fft_vals)
    fft_freq   = np.fft.rfftfreq(len(segment), d=1 / sampling_rate)


    min_len   = min(len(magnitude), len(fft_freq))
    magnitude = magnitude[:min_len]
    fft_freq  = fft_freq[:min_len]

    features["peak_freq"]      = fft_freq[np.argmax(magnitude)]
    features["spectral_energy"] = np.sum(magnitude ** 2)

    return features

In [ ]:
def get_label(filepath):
    name = Path(filepath).stem.upper()
    if name.startswith("NORMAL") or name.isdigit():
        return "normal"
    elif any(x in name for x in ["IR", "OR", "B", "BALL"]):
        return "fault"
    else:
        return "normal"  



In [181]:
def build_dataset(data_dir, window_size=2048, stride=512):
   
    dataset = []
    skipped = 0

    for root, _, files in os.walk(data_dir):
        for file in files:
            if not file.endswith(".mat"):
                continue

            filepath = os.path.join(root, file)

            try:
                signal = load_signal(filepath)

                if signal is None or len(signal) == 0:
                    print(f"Skipped (no signal): {file}")
                    skipped += 1
                    continue

                segments = segment_signal(signal, window_size, stride)

                if len(segments) == 0:
                    print(f"Skipped (too short to segment): {file}")
                    skipped += 1
                    continue

                label = get_label(filepath)

                for seg in segments:
                    features         = extract_features(seg)
                    features["label"] = label
                    features["file"]  = file
                    dataset.append(features)

            except Exception as e:
                print(f"Error in {file}: {e}")
                skipped += 1

    print(f"\nSkipped files: {skipped}")
    print(f"Total windows: {len(dataset)}")
    return pd.DataFrame(dataset)

In [182]:
df = build_dataset(DATA_DIR)

print("\nClass distribution:")
print(df["label"].value_counts())

print("\nDataset shape:", df.shape)
df.head()


Skipped files: 0
Total windows: 79206

Class distribution:
label
normal    69739
fault      9467
Name: count, dtype: int64

Dataset shape: (79206, 10)


,mean,std,rms,peak,crest_factor,kurtosis,peak_freq,spectral_energy,label,file
0,0.010940,0.419304,0.419447,1.652651,3.940074,0.801974,720.703125,369214.581921,normal,149.mat
1,0.012740,0.400557,0.400760,1.531237,3.820835,0.463263,720.703125,337160.651108,normal,149.mat
2,0.011324,0.450407,0.450549,1.555436,3.452311,0.545401,714.843750,425979.653279,normal,149.mat
3,0.011361,0.472019,0.472156,1.555436,3.294330,0.482054,703.125000,467790.541021,normal,149.mat
4,0.009512,0.476265,0.476360,1.611971,3.383935,0.439750,708.984375,476072.875685,normal,149.mat


In [183]:
df_supervised = df[df["label"] != "unknown"]
df_unsupervised = df[df["label"] == "unknown"]

print("Supervised:", df_supervised.shape)
print("Unsupervised:", df_unsupervised.shape)

Supervised: (79206, 10)
Unsupervised: (0, 10)


## Model Fine-Tuning on Subset of Data

To evaluate model performance and simulate limited-data scenarios, we train a classification model using a subset of the dataset.


In [184]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report

In [ ]:

df_subset = df.sample(frac=0.3, random_state=42)

feature_cols = [c for c in df.columns if c not in ["label", "file"]]
X = df_subset[feature_cols].values
labels = df_subset["label"].values

# Train
X_train, X_test, y_train, y_test = train_test_split(
    X, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")


X_train_normal = X_train[y_train == "normal"]
model = IsolationForest(contamination=0.05, random_state=42)
model.fit(X_train_normal)


preds = model.predict(X_test)
preds_binary = (preds == -1).astype(int)
y_true = (y_test == "fault").astype(int)

print(classification_report(y_true, preds_binary,
      target_names=["normal", "fault"]))

Train: 19009 samples
Test:  4753 samples
              precision    recall  f1-score   support

      normal       0.88      0.95      0.91      4188
       fault       0.03      0.01      0.02       565

    accuracy                           0.84      4753
   macro avg       0.45      0.48      0.46      4753
weighted avg       0.78      0.84      0.80      4753

